# BO Forge cost-aware human-review campaign

This notebook demonstrates the v0.4.3 `CampaignSession` workflow for a cost-aware, review-gated mixed-variable campaign.

The design reference is `/Users/liangze/Desktop/bo_forge/PyTorch & BoTorch/Part 5/tutorial_04_budget_aware_and_human_in_the_loop_bo_workflows_worked.ipynb`, especially acquisition-minus-cost utility, cumulative-cost tracking, and accepted/rejected workflow history.

## 1. Setup

The CSV log includes review columns and cost columns only because the YAML config enables both `review` and `cost`.

In [ ]:
from pathlib import Path
import os
import shutil
import sys

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

mpl_cache = PROJECT_ROOT / ".matplotlib-cache"
mpl_cache.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_cache))

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from bo_forge.costs import evaluate_cost
from bo_forge.session import CampaignSession

In [ ]:
config_path = PROJECT_ROOT / "configs" / "07_cost_aware_human_review_logei.yaml"
seed_log_path = PROJECT_ROOT / "examples" / "07_cost_aware_human_review_campaign_log.csv"
working_log_path = PROJECT_ROOT / "examples" / "07_cost_aware_human_review_working_log.csv"
latest_suggestion_path = PROJECT_ROOT / "examples" / "07_cost_aware_human_review_latest_suggestions.csv"
report_path = PROJECT_ROOT / "reports" / "07_cost_aware_human_review_report.txt"
progress_path = PROJECT_ROOT / "reports" / "07_cost_aware_human_review_progress.pdf"
cost_progress_path = PROJECT_ROOT / "reports" / "07_cost_aware_human_review_cost_progress.pdf"
TARGET_OBSERVED_ROWS = 15

shutil.copyfile(seed_log_path, working_log_path)

campaign = CampaignSession.from_files(config_path=config_path, log_path=working_log_path)

## 2. Inspect state, review queue, and budget

In [ ]:
campaign.validate()

display(campaign.summary())
display(campaign.cost_summary())
display(campaign.review_queue())
display(campaign.next_action())
campaign.df

## 3. Synthetic objective and actual cost

The simulator is deterministic. `actual_cost` is also deterministic here, but in a real campaign it would be entered after the experiment is run.

In [ ]:
def simulate_yield(row):
    loading = float(row["catalyst_loading"])
    reaction_time = int(row["reaction_time"])
    time_scaled = (reaction_time - 10) / 50
    base = float(row["base_equivalents"])
    solvent = row["solvent"]

    solvent_bonus = {"MeCN": 5.0, "EtOH": 2.0, "Water": 0.5}[solvent]
    base_bonus = {0.1: -4.0, 0.2: 2.5, 0.5: 5.0, 1.0: 1.0}[base]
    peak = np.exp(
        -0.5
        * (
            ((loading - 0.12) / 0.035) ** 2
            + ((time_scaled - 0.55) / 0.22) ** 2
        )
    )
    interaction = 3.0 * np.sin(10.0 * loading + 2.0 * time_scaled)
    return float(45.0 + 25.0 * peak + interaction + base_bonus + solvent_bonus)


def simulate_actual_cost(row):
    return 1.05 * evaluate_cost(campaign.config, row)

## 4. Fill the initial design through review

Initial Sobol suggestions include `cost_estimate` and pending review status, but leave `utility` blank because no model acquisition exists yet.

In [ ]:
initial_suggestions = campaign.suggest_next(batch_size=2)
initial_suggestions.to_csv(latest_suggestion_path, index=False)

display(initial_suggestions)
display(campaign.suggestion_quality(initial_suggestions))

campaign.append_suggestions(initial_suggestions)
display(campaign.review_queue())
display(campaign.next_action())

campaign.review_suggestion(str(initial_suggestions.loc[0, "row_id"]), "accept", "run first")
campaign.review_suggestion(str(initial_suggestions.loc[1, "row_id"]), "reject", "too inconvenient")
campaign.mark_observed(
    row_id=str(initial_suggestions.loc[0, "row_id"]),
    objective_value=simulate_yield(initial_suggestions.loc[0]),
    actual_cost=simulate_actual_cost(initial_suggestions.loc[0]),
)

display(campaign.summary())
display(campaign.cost_summary())
campaign.df.tail(4)

In [ ]:
final_initial = campaign.suggest_next(batch_size=1)
display(final_initial)

campaign.append_suggestions(final_initial)
campaign.review_suggestion(str(final_initial.loc[0, "row_id"]), "accept", "complete initial design")
campaign.mark_observed(
    row_id=str(final_initial.loc[0, "row_id"]),
    objective_value=simulate_yield(final_initial.loc[0]),
    actual_cost=simulate_actual_cost(final_initial.loc[0]),
)

display(campaign.summary())
campaign.df.tail(4)

## 5. Request cost-aware BO suggestions

Model-based cost-aware suggestions use `utility = acquisition - cost.weight * cost_estimate`. For batches, v0.4.3 uses greedy single-candidate utility rather than joint cost-aware qLogEI.

In [ ]:
bo_suggestions = campaign.suggest_next(batch_size=2)
bo_suggestions.to_csv(latest_suggestion_path, index=False)

display(bo_suggestions)
display(campaign.suggestion_quality(bo_suggestions))

## 6. Review, run one accepted experiment, and resume

Rejected and deferred suggestions remain in the CSV for auditability, but they do not reserve budget or block the next suggestion.

In [ ]:
campaign.append_suggestions(bo_suggestions)
display(campaign.review_queue())
display(campaign.next_action())

campaign.review_suggestion(str(bo_suggestions.loc[0, "row_id"]), "accept", "best utility")
campaign.review_suggestion(str(bo_suggestions.loc[1, "row_id"]), "defer", "hold for later")
campaign.mark_observed(
    row_id=str(bo_suggestions.loc[0, "row_id"]),
    objective_value=simulate_yield(bo_suggestions.loc[0]),
    actual_cost=simulate_actual_cost(bo_suggestions.loc[0]),
)

campaign.reload()
display(campaign.summary())
display(campaign.cost_summary())
display(campaign.review_queue())
campaign.df.tail(6)

## 7. Export report and cost-progress plot

In [ ]:
target_records = []
while len(campaign.observed_data()) < TARGET_OBSERVED_ROWS:
    remaining = TARGET_OBSERVED_ROWS - len(campaign.observed_data())
    batch_size = min(campaign.config.bo.batch_size, remaining)
    suggestions = campaign.suggest_next(batch_size=batch_size)
    suggestions.to_csv(latest_suggestion_path, index=False)
    campaign.append_suggestions(suggestions)

    for _, suggestion in suggestions.iterrows():
        campaign.review_suggestion(str(suggestion["row_id"]), "accept", "target campaign run")
        value = simulate_yield(suggestion)
        actual_cost = simulate_actual_cost(suggestion)
        campaign.mark_observed(
            row_id=str(suggestion["row_id"]),
            objective_value=value,
            actual_cost=actual_cost,
        )
        target_records.append(
            {
                "source": suggestion["source"],
                "review_status": "accepted",
                "solvent": suggestion["solvent"],
                "cost_estimate": float(suggestion["cost_estimate"]),
                "cost_actual": actual_cost,
                "yield_score": value,
            }
        )
    campaign.reload()

assert len(campaign.observed_data()) == TARGET_OBSERVED_ROWS

if target_records:
    display(pd.DataFrame(target_records))

display(campaign.summary())
display(campaign.cost_summary())
display(campaign.review_queue())
display(campaign.next_action())

In [ ]:
report = campaign.report()
campaign.export_report(report_path)
campaign.plot_progress(save_path=progress_path)
campaign.plot_cost_progress(save_path=cost_progress_path)

display(report["summary"])
display(report["cost_summary"])
print(report_path)
print(progress_path)
print(cost_progress_path)